In [2]:
import sys
from pathlib import Path

import pandas as pd

# Layihənin əsas qovluğu
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Notebook daxilindən db.py faylına çıxış
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

from db import get_engine

# CSV fayllarının yerləşdiyi qovluq
DATA_DIR = PROJECT_ROOT / "data" / "bank_data"

customers = pd.read_csv(DATA_DIR / "customers.csv")
loans = pd.read_csv(DATA_DIR / "loans.csv")
transactions = pd.read_csv(DATA_DIR / "transactions.csv")

print("Customers:", customers.shape)
print("Loans:", loans.shape)
print("Transactions:", transactions.shape)

Customers: (5000, 13)
Loans: (4820, 10)
Transactions: (77396, 8)


In [4]:
from sqlalchemy import text

# SQL Server bağlantısı
engine = get_engine()

# CSV məlumatlarını Bronze cədvəllərinə yükləyirik
customers.to_sql(
    "customers",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    chunksize=1000
)

loans.to_sql(
    "loans",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    chunksize=1000
)

transactions.to_sql(
    "transactions",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Bronze məlumatları SQL Server-ə yükləndi!")

# SQL Server-də sətir saylarını yoxlayırıq
with engine.connect() as conn:
    for table in ["customers", "loans", "transactions"]:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM bronze.{table}")
        ).scalar()

        print(f"bronze.{table}: {count:,} sətir")

Bronze məlumatları SQL Server-ə yükləndi!
bronze.customers: 5,000 sətir
bronze.loans: 4,820 sətir
bronze.transactions: 77,396 sətir
